# M4 — Fine-tuning VGG16 sur Colab (images locales)

Reprise du meilleur checkpoint M4. Bloc 5 dégelé, LR 1e-5, 5 epochs max, patience 2. Les CSV restent sur Drive, les images sont copiées vers `/content` pour accélérer l'entraînement. Résultats sauvegardés sur Drive. Aucun accès au test.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil, time, json, zipfile
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

tf.keras.utils.set_random_seed(42)
PROJECT_DIR = Path('/content/drive/MyDrive/Apprentissage/DS/projet')
DRIVE_DATASET_ROOT = PROJECT_DIR / 'COVID-19_Radiography_Dataset'
DATASET_ROOT = Path('/content/COVID-19_Radiography_Dataset')
OUTPUT_DIR = PROJECT_DIR / 'M4_finetuning'
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT = PROJECT_DIR / 'M4_VGG16_multiclass_best.keras'
print('GPU :', tf.config.list_physical_devices('GPU'))

# Copie une fois par session Colab. Le disque /content est temporaire.
if not DATASET_ROOT.exists():
    start = time.perf_counter()
    shutil.copytree(DRIVE_DATASET_ROOT, DATASET_ROOT)
    print(f'Copie locale terminée en {(time.perf_counter()-start)/60:.1f} min')
else:
    print('Dataset local déjà présent :', DATASET_ROOT)


In [ ]:
train_df = pd.read_csv(PROJECT_DIR / 'train.csv')
val_df = pd.read_csv(PROJECT_DIR / 'validation.csv')

# Les chemins CSV sont ceux du PC Windows ; on conserve seulement la partie
# après COVID-19_Radiography_Dataset, comme dans E2/E3 du binôme.
def local_filepath(original):
    normalized = str(original).replace('\\', '/')
    marker = 'COVID-19_Radiography_Dataset/'
    if marker not in normalized:
        raise ValueError(f'Chemin CSV inattendu : {original}')
    return str(DATASET_ROOT / normalized.split(marker, 1)[1])

train_df['filepath'] = train_df['filepath'].map(local_filepath)
val_df['filepath'] = val_df['filepath'].map(local_filepath)
# Deux vérifications rapides pour éviter un fit avec 0 image.
assert Path(train_df['filepath'].iloc[0]).is_file(), train_df['filepath'].iloc[0]
assert Path(val_df['filepath'].iloc[0]).is_file(), val_df['filepath'].iloc[0]
print('Train :', len(train_df), 'Validation :', len(val_df))


In [ ]:
classes = sorted(train_df['class'].dropna().unique().tolist())
train_df['class'] = train_df['class'].astype(str)
val_df['class'] = val_df['class'].astype(str)
datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
train_generator = datagen.flow_from_dataframe(train_df, x_col='filepath', y_col='class', classes=classes,
    target_size=(224,224), color_mode='rgb', class_mode='categorical', batch_size=32, shuffle=True, seed=42)
val_generator = datagen.flow_from_dataframe(val_df, x_col='filepath', y_col='class', classes=classes,
    target_size=(224,224), color_mode='rgb', class_mode='categorical', batch_size=32, shuffle=False)
print('Classes :', val_generator.class_indices)

In [ ]:
# Essai 1 : dernier bloc VGG16 uniquement ; toujours repartir du checkpoint original.
EXPERIMENT_NAME = 'M4_FT_B5_LR1e-5'
LEARNING_RATE = 1e-5
EPOCHS = 5
UNFREEZE_BLOCK = 5

# Compatibilité checkpoint Keras : anciens paramètres GlorotUniform non reconnus
# par la version Keras actuelle de Colab. Copie corrigée, poids inchangés.
compatible_checkpoint = OUTPUT_DIR / (CHECKPOINT.stem + '_compatible.keras')
if not compatible_checkpoint.exists():
    def clean_initializer_config(obj):
        if isinstance(obj, dict):
            if obj.get('class_name') == 'GlorotUniform':
                cfg = obj.get('config', {})
                cfg.pop('input_axes', None)
                cfg.pop('output_axes', None)
            for value in obj.values():
                clean_initializer_config(value)
        elif isinstance(obj, list):
            for value in obj:
                clean_initializer_config(value)
    with zipfile.ZipFile(CHECKPOINT, 'r') as source:
        config = json.loads(source.read('config.json'))
        clean_initializer_config(config)
        with zipfile.ZipFile(compatible_checkpoint, 'w') as dest:
            for member in source.infolist():
                payload = (json.dumps(config).encode('utf-8') if member.filename == 'config.json'
                           else source.read(member.filename))
                dest.writestr(member, payload)

model = load_model(compatible_checkpoint, compile=False)
base_model = model.layers[0]  # VGG16 dans Sequential
base_model.trainable = True
for layer in base_model.layers:
    layer.trainable = layer.name.startswith(f'block{UNFREEZE_BLOCK}_')
print('Couches VGG16 entraînables :', [l.name for l in base_model.layers if l.trainable])

model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
              loss='categorical_crossentropy', metrics=['accuracy'])
BEST_PATH = OUTPUT_DIR / f'{EXPERIMENT_NAME}_best.keras'
callbacks = [
    EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
    ModelCheckpoint(str(BEST_PATH), monitor='val_loss', save_best_only=True),
    CSVLogger(str(OUTPUT_DIR / f'{EXPERIMENT_NAME}_history.csv')),
]


In [ ]:
history = model.fit(train_generator, validation_data=val_generator,
                    epochs=EPOCHS, callbacks=callbacks)
print('Meilleur checkpoint :', BEST_PATH)

In [ ]:
best_model = load_model(BEST_PATH)
val_generator.reset()
y_prob = best_model.predict(val_generator, verbose=1)
y_true = val_generator.classes
y_pred = np.argmax(y_prob, axis=1)
report = classification_report(y_true, y_pred, target_names=classes, output_dict=True, zero_division=0)
pd.DataFrame(report).T.to_csv(OUTPUT_DIR / f'{EXPERIMENT_NAME}_classification_report.csv')
pd.DataFrame({'y_true': y_true, 'y_pred': y_pred}).to_csv(OUTPUT_DIR / f'{EXPERIMENT_NAME}_predictions.csv', index=False)
print(classification_report(y_true, y_pred, target_names=classes, digits=4, zero_division=0))
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=classes).plot(cmap='Blues', xticks_rotation=35)
plt.title(EXPERIMENT_NAME + ' — validation')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'{EXPERIMENT_NAME}_confusion_matrix.png', dpi=160, bbox_inches='tight')
plt.show()
print('Recall COVID :', report['COVID']['recall'], 'F1 COVID :', report['COVID']['f1-score'])

In [ ]:
hist = pd.read_csv(OUTPUT_DIR / f'{EXPERIMENT_NAME}_history.csv')
for metric in ('loss', 'accuracy'):
    plt.figure(figsize=(7, 4))
    plt.plot(hist[metric], label='Train')
    plt.plot(hist['val_' + metric], label='Validation')
    plt.title(f'{EXPERIMENT_NAME} — {metric}')
    plt.xlabel('Epoch')
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'{EXPERIMENT_NAME}_{metric}.png', dpi=160)
    plt.show()